In [ ]:
from models import InitializationAngle
from utils import GraspAnalysisUtils
from pathlib import Path
import numpy as np

In [ ]:
# Find folder containing CSV files for analysis
data_file = GraspAnalysisUtils.select_file()

In [ ]:
# Get initialization variables
starting_angle, increment, trials = GraspAnalysisUtils.get_initialization_parameters()

In [ ]:
# Assignment of variables
SAMPLE_SIZE = 200
MIN_FORCE_THRESHOLD = 10
FS = 1000 # Data collection rate

In [ ]:
filename = Path(data_file).name

force_data = GraspAnalysisUtils.load_and_preprocess_data(data_file)

grasps = GraspAnalysisUtils.detect_grasp_regions(force_data, SAMPLE_SIZE, MIN_FORCE_THRESHOLD)
number_of_grasps = len(grasps)

if number_of_grasps == 0:
    print(f"No grasps detected in file: {filename}")
else:
    rolling_avg = []
    rolling_std = []
    rolling_median = []

    avg_forces = []
    for grasp in grasps:
        grasp = GraspAnalysisUtils.calculate_grasp_force(force_data[grasp.start_idx:grasp.end_idx], grasp, FS)

        avg_forces.append(grasp.avg_force)
        rolling_avg.append(np.mean(avg_forces))
        rolling_std.append(np.std(avg_forces))
        rolling_median.append(np.median(avg_forces))

    max_force, max_angle = GraspAnalysisUtils.analyze_initialization_angles(avg_forces, starting_angle, increment, trials)

    GraspAnalysisUtils.report_initialization_results(filename, number_of_grasps, max_force, max_angle)